In [2]:
import pdfplumber
import re
import pandas as pd

data = []

with pdfplumber.open("2004 (Vol II).pdf") as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        
        if not text:
            continue
        
        # Extract State
        state_match = re.search(r"STATE/UT\s*:\s*(.*)", text)
        state = state_match.group(1).strip() if state_match else None
        
        # Extract Constituency
        const_match = re.search(r"CONSTITUENCY\s*:\s*(.*)", text)
        constituency = const_match.group(1).strip() if const_match else None
        
        # ELECTORS TOTAL (Men Women Total)
        electors_match = re.search(r"TOTAL\s+(\d+)\s+(\d+)\s+(\d+)", text)
        
        # VOTERS GENERAL (Men Women Total)
        voters_match = re.search(r"III\.\s*VOTERS.*?GENERAL\s+(\d+)\s+(\d+)\s+(\d+)", text, re.DOTALL)
        
        if electors_match and voters_match:
            data.append({
                "State": state,
                "Constituency": constituency,
                "Male_Electors": int(electors_match.group(1)),
                "Female_Electors": int(electors_match.group(2)),
                "Total_Electors": int(electors_match.group(3)),
                "Male_Voters": int(voters_match.group(1)),
                "Female_Voters": int(voters_match.group(2)),
                "Total_Voters": int(voters_match.group(3))
            })

# Convert to DataFrame
df = pd.DataFrame(data)

# Save CSV
df.to_csv("dataset_2004.csv", index=False)

print("Extraction Completed ✅")

Extraction Completed ✅


In [10]:
import pdfplumber
import re
import pandas as pd

pdf_path = "Constituency data summary 2014.pdf"

data = []

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()

        if not text:
            continue

        # -----------------------------
        # 🔹 Extract State Code (S01 etc.)
        # -----------------------------
        state_match = re.search(r"State/UT\s*:\s*(S\d+)", text)
        state_code = state_match.group(1) if state_match else "UNKNOWN"

        # -----------------------------
        # 🔹 Extract Constituency
        # -----------------------------
        const_match = re.search(r"Constituency\s*:\s*(.*)", text)
        constituency = const_match.group(1).strip() if const_match else None

        # -----------------------------
        # 🔹 Extract ELECTORS TOTAL
        # Example:
        # 3. TOTAL 553276 577935 1131211
        # -----------------------------
        electors_match = re.search(
            r"ELECTORS[\s\S]*?TOTAL\s+(\d+)\s+(\d+)\s+(\d+)",
            text
        )

        # -----------------------------
        # 🔹 Extract VOTERS GENERAL
        # Example:
        # 1. GENERAL 426628 436702 863330
        # -----------------------------
        voters_match = re.search(
            r"VOTERS[\s\S]*?GENERAL\s+(\d+)\s+(\d+)\s+(\d+)",
            text
        )

        # -----------------------------
        # 🔹 Debug (optional)
        # -----------------------------
        if not electors_match or not voters_match:
            print(f"⚠️ Skipped page {i+1}")
            continue

        # -----------------------------
        # 🔹 Append Clean Data
        # -----------------------------
        data.append({
            "State_Code": state_code,
            "Constituency": constituency,
            "Male_Electors": int(electors_match.group(1)),
            "Female_Electors": int(electors_match.group(2)),
            "Total_Electors": int(electors_match.group(3)),
            "Male_Voters": int(voters_match.group(1)),
            "Female_Voters": int(voters_match.group(2)),
            "Total_Voters": int(voters_match.group(3)),
        })

# -----------------------------
# 🔹 Convert to DataFrame
# -----------------------------
df = pd.DataFrame(data)

# -----------------------------
# 🔹 Save CSV
# -----------------------------
df.to_csv("dataset_2014.csv", index=False)

# -----------------------------
# 🔹 Final Output
# -----------------------------
print("\n✅ Extraction Completed!")
print("Total Rows Extracted:", len(df))

⚠️ Skipped page 544

✅ Extraction Completed!
Total Rows Extracted: 543


In [11]:
import pdfplumber
import re
import pandas as pd

pdf_path = "32. Constituency Data Summery Report 2019.pdf"

data = []

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()

        if not text:
            continue

        # -----------------------------
        # 🔹 Extract State Code
        # -----------------------------
        state_match = re.search(r"STATE/UT\s*:\s*(S\d+)", text, re.IGNORECASE)
        state_code = state_match.group(1) if state_match else "UNKNOWN"

        # -----------------------------
        # 🔹 Extract Constituency
        # -----------------------------
        const_match = re.search(r"CONSTITUENCY\s*:\s*(.*)", text, re.IGNORECASE)
        constituency = const_match.group(1).strip() if const_match else None

        # -----------------------------
        # 🔹 ELECTORS TOTAL (4 values)
        # MEN WOMEN THIRD TOTAL
        # -----------------------------
        electors_match = re.search(
            r"ELECTORS[\s\S]*?TOTAL\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)",
            text,
            re.IGNORECASE
        )

        # -----------------------------
        # 🔹 VOTERS GENERAL (4 values)
        # -----------------------------
        voters_match = re.search(
            r"VOTERS[\s\S]*?GENERAL\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)",
            text,
            re.IGNORECASE
        )

        if not electors_match or not voters_match:
            print(f"⚠️ Skipped page {i+1}")
            continue

        data.append({
            "State_Code": state_code,
            "Constituency": constituency,
            "Male_Electors": int(electors_match.group(1)),
            "Female_Electors": int(electors_match.group(2)),
            "Other_Electors": int(electors_match.group(3)),
            "Total_Electors": int(electors_match.group(4)),
            "Male_Voters": int(voters_match.group(1)),
            "Female_Voters": int(voters_match.group(2)),
            "Other_Voters": int(voters_match.group(3)),
            "Total_Voters": int(voters_match.group(4)),
        })

# -----------------------------
# 🔹 Convert to DataFrame
# -----------------------------
df = pd.DataFrame(data)

# -----------------------------
# 🔹 Add Year
# -----------------------------
df["Year"] = 2019

# -----------------------------
# 🔹 Save CSV
# -----------------------------
df.to_csv("dataset_2019.csv", index=False)

print("\n✅ 2019 Extraction Completed!")
print("Rows:", len(df))

⚠️ Skipped page 2
⚠️ Skipped page 4
⚠️ Skipped page 6
⚠️ Skipped page 8
⚠️ Skipped page 10
⚠️ Skipped page 12
⚠️ Skipped page 14
⚠️ Skipped page 16
⚠️ Skipped page 18
⚠️ Skipped page 20
⚠️ Skipped page 22
⚠️ Skipped page 24
⚠️ Skipped page 26
⚠️ Skipped page 28
⚠️ Skipped page 30
⚠️ Skipped page 32
⚠️ Skipped page 34
⚠️ Skipped page 36
⚠️ Skipped page 38
⚠️ Skipped page 40
⚠️ Skipped page 42
⚠️ Skipped page 44
⚠️ Skipped page 46
⚠️ Skipped page 48
⚠️ Skipped page 50
⚠️ Skipped page 52
⚠️ Skipped page 54
⚠️ Skipped page 56
⚠️ Skipped page 58
⚠️ Skipped page 60
⚠️ Skipped page 62
⚠️ Skipped page 64
⚠️ Skipped page 66
⚠️ Skipped page 68
⚠️ Skipped page 70
⚠️ Skipped page 72
⚠️ Skipped page 74
⚠️ Skipped page 76
⚠️ Skipped page 78
⚠️ Skipped page 80
⚠️ Skipped page 82
⚠️ Skipped page 84
⚠️ Skipped page 86
⚠️ Skipped page 88
⚠️ Skipped page 90
⚠️ Skipped page 92
⚠️ Skipped page 94
⚠️ Skipped page 96
⚠️ Skipped page 98
⚠️ Skipped page 100
⚠️ Skipped page 102
⚠️ Skipped page 104
⚠️ Skipped pa

In [12]:
import pdfplumber
import re
import pandas as pd

pdf_path = "32-Constituency_data_summery_2024.pdf"

data = []

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()

        if not text:
            continue

        # -----------------------------
        # 🔹 Extract State Name
        # -----------------------------
        state_match = re.search(r"STATE/UT\s*:\s*(.*)", text, re.IGNORECASE)
        state = state_match.group(1).split("CODE")[0].strip() if state_match else None

        # -----------------------------
        # 🔹 Extract State Code
        # -----------------------------
        code_match = re.search(r"CODE\s*:\s*(S\d+)", text)
        state_code = code_match.group(1) if code_match else "UNKNOWN"

        # -----------------------------
        # 🔹 Extract Constituency
        # -----------------------------
        const_match = re.search(r"CONSTITUENCY\s*:\s*(.*)", text, re.IGNORECASE)
        constituency = const_match.group(1).strip() if const_match else None

        # -----------------------------
        # 🔹 ELECTORS TOTAL
        # MEN WOMEN THIRD TOTAL
        # -----------------------------
        electors_match = re.search(
            r"ELECTORS[\s\S]*?TOTAL\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)",
            text,
            re.IGNORECASE
        )

        # -----------------------------
        # 🔹 VOTERS GENERAL
        # -----------------------------
        voters_match = re.search(
            r"VOTERS[\s\S]*?GENERAL\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)",
            text,
            re.IGNORECASE
        )

        if not electors_match or not voters_match:
            print(f"⚠️ Skipped page {i+1}")
            continue

        data.append({
            "State": state,
            "State_Code": state_code,
            "Constituency": constituency,
            "Male_Electors": int(electors_match.group(1)),
            "Female_Electors": int(electors_match.group(2)),
            "Other_Electors": int(electors_match.group(3)),
            "Total_Electors": int(electors_match.group(4)),
            "Male_Voters": int(voters_match.group(1)),
            "Female_Voters": int(voters_match.group(2)),
            "Other_Voters": int(voters_match.group(3)),
            "Total_Voters": int(voters_match.group(4)),
        })

# -----------------------------
# 🔹 Convert to DataFrame
# -----------------------------
df = pd.DataFrame(data)

# -----------------------------
# 🔹 Add Year
# -----------------------------
df["Year"] = 2024

# -----------------------------
# 🔹 Save CSV
# -----------------------------
df.to_csv("dataset_2024.csv", index=False)

print("\n✅ 2024 Extraction Completed!")
print("Rows:", len(df))

⚠️ Skipped page 2
⚠️ Skipped page 4
⚠️ Skipped page 6
⚠️ Skipped page 8
⚠️ Skipped page 10
⚠️ Skipped page 12
⚠️ Skipped page 14
⚠️ Skipped page 16
⚠️ Skipped page 18
⚠️ Skipped page 20
⚠️ Skipped page 22
⚠️ Skipped page 24
⚠️ Skipped page 26
⚠️ Skipped page 28
⚠️ Skipped page 30
⚠️ Skipped page 32
⚠️ Skipped page 34
⚠️ Skipped page 36
⚠️ Skipped page 38
⚠️ Skipped page 40
⚠️ Skipped page 42
⚠️ Skipped page 44
⚠️ Skipped page 46
⚠️ Skipped page 48
⚠️ Skipped page 50
⚠️ Skipped page 52
⚠️ Skipped page 54
⚠️ Skipped page 56
⚠️ Skipped page 58
⚠️ Skipped page 60
⚠️ Skipped page 62
⚠️ Skipped page 64
⚠️ Skipped page 66
⚠️ Skipped page 68
⚠️ Skipped page 70
⚠️ Skipped page 72
⚠️ Skipped page 74
⚠️ Skipped page 76
⚠️ Skipped page 78
⚠️ Skipped page 80
⚠️ Skipped page 82
⚠️ Skipped page 84
⚠️ Skipped page 86
⚠️ Skipped page 88
⚠️ Skipped page 90
⚠️ Skipped page 92
⚠️ Skipped page 94
⚠️ Skipped page 96
⚠️ Skipped page 98
⚠️ Skipped page 100
⚠️ Skipped page 102
⚠️ Skipped page 104
⚠️ Skipped pa